# Modèle de prévision - hausse de loyer 2026 (baseline : modèle D)

**Cible (définition de la section 1 du notebook final) :** croissance annualisée du **loyer effectif** à unité constante,
médiane de tous les baux débutant dans l'année (renouvellements et relocations, six immeubles).

**Modèle de référence (baseline) :** un gradient boosting (scikit-learn) entraîné **au niveau des paires de baux**.
Il suit littéralement la définition : il prédit la hausse de **chaque bail** de l'année à partir de son bail précédent,
puis prend la **médiane**. Ce notebook sert de base pour améliorer ce modèle; chaque amélioration sera jugée sur le même
backtest (2023, 2024, 2025).

### Comment nos analyses se relient dans le modèle

| Notebook | Ce qu'il a établi | Rôle dans le modèle |
|---|---|---|
| `equinoxe-2026` (section 1) | la définition de la hausse | la **cible** : médiane des hausses de chaque bail |
| `exploration` / section 2 | `sRentEffective` est fiable; clé d'unité; colonnes | les **données** utilisées sont valides |
| `unite_constante` | apparier, filtrer, annualiser | les **paires** sur lesquelles le modèle apprend |
| `concessions` | effectif = contractuel - hausse du rabais | la variable **rabais du bail précédent** |
| `renouvellements_relocations` | renouvellements et relocations suivent des règles différentes; The Met relève de l'Ontario | les variables **type de bail** et **taux réglementaire par province** |
| `donnees_publiques` + taux réglementaires | SCHL, TAL, Ontario | les **variables externes** testées |

## 0. Configuration et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

LEASES_PATH = "private_data/equinoxe_lease_history.csv"
CMHC_PATH = "public_data/cmhc_market_data.csv"
REGULATORY_RATES_PATH = "public_data/regulatory_rates.csv"

UNIT_KEY = ["sPropCode", "sUnitCode"]
MIN_GAP_YEARS = 0.5
MAX_GAP_YEARS = 2.5
DAYS_PER_YEAR = 365.25
PCT = 100
RENEWAL, TURNOVER = 1, 0
ONTARIO = "Ontario"
MONTREAL_MARKET = "Montréal"

FIRST_HISTORY_YEAR = 2019
BACKTEST_YEARS = [2023, 2024, 2025]
FORECAST_YEAR = 2026
OTTAWA_MARKET = "Ottawa — partie Ontario"

# Modèle D (gradient boosting au niveau des paires)
UNIT_FEATURES = ["sBeds", "sSqft", "sFloor", "gap_years", "is_renewal"]
GBM_PARAMS = {
    "loss": "absolute_error",   # erreur absolue : le modèle vise la médiane, comme notre définition
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_depth": 3,             # arbres peu profonds : peu de données par année
    "min_samples_leaf": 40,     # chaque feuille repose sur au moins 40 paires
    "random_state": 0,
}
MODEL_D_PATH = Path("models/modele_d_gbm.joblib")

LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sRenewal": "is_renewal",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sTermMonths": "lease_term_months",
}

leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo"]).rename(columns=LEASE_COLUMN_RENAMES)
leases["lease_start_year"] = leases["lease_start"].dt.year
cmhc = pd.read_csv(CMHC_PATH)
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
print(f"{len(leases):,} baux | SCHL : {cmhc['year'].min()}-{cmhc['year'].max()} | taux réglementaires : "
      f"{regulatory_rates.index.min()}-{regulatory_rates.index.max()}")

## 1. Construire les paires de baux et l'historique de la cible

On reprend la méthode de `unite_constante` : chaque bail est comparé au bail précédent de la même unité, les écarts non
comparables sont filtrés, la variation est annualisée. Pour chaque paire, on calcule aussi la **hausse du rabais**
(rabais du nouveau bail - rabais du bail précédent, en points), mécanisme établi dans `concessions`.

Chaque paire est rangée dans un **segment** :
- `renewal_QC` : renouvellement dans un immeuble québécois (encadré par le TAL);
- `renewal_ON` : renouvellement à The Met (Ontario, probablement exempté du plafond);
- `turnover` : relocation (loyer de marché, sans plafond dans les deux provinces).

In [ ]:
def build_comparable_pairs(lease_df):
    """Paires de baux consécutifs d'une même unité, filtrées, avec croissances annualisées et hausse du rabais."""
    ordered = lease_df.sort_values(UNIT_KEY + ["lease_start"]).copy()
    previous = (ordered.groupby(UNIT_KEY)[["contract_rent", "effective_rent", "lease_start"]]
                .shift(1).add_prefix("prev_"))
    pairs = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    pairs["gap_years"] = (pairs["lease_start"] - pairs["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    is_comparable = pairs["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    pairs = pairs[is_comparable].copy()

    for rent in ["contract_rent", "effective_rent"]:
        ratio = pairs[rent] / pairs[f"prev_{rent}"]
        pairs[f"{rent}_growth_pct"] = (ratio ** (1 / pairs["gap_years"]) - 1) * PCT

    discount_now = (1 - pairs["effective_rent"] / pairs["contract_rent"]) * PCT
    discount_before = (1 - pairs["prev_effective_rent"] / pairs["prev_contract_rent"]) * PCT
    pairs["discount_change_pts"] = discount_now - discount_before
    pairs["prev_discount_pct"] = discount_before

    is_ontario = pairs["sState"] == ONTARIO
    pairs["segment"] = np.select(
        [pairs["is_renewal"].eq(RENEWAL) & ~is_ontario, pairs["is_renewal"].eq(RENEWAL) & is_ontario],
        ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


pairs = build_comparable_pairs(leases)
pairs = pairs[pairs["lease_start_year"] >= FIRST_HISTORY_YEAR]
print(f"{len(pairs):,} paires comparables depuis {FIRST_HISTORY_YEAR}")

In [ ]:
yearly_target = pairs.groupby("lease_start_year").agg(
    pairs=("effective_rent_growth_pct", "size"),
    effective_growth=("effective_rent_growth_pct", "median"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median"))

yearly_segments = pairs.groupby(["lease_start_year", "segment"]).agg(
    pairs=("effective_rent_growth_pct", "size"),
    contract_growth=("contract_rent_growth_pct", "median"),
    discount_change=("discount_change_pts", "median")).unstack("segment")

print("Cible et composantes, tous baux :")
print(yearly_target.round(2), end="\n\n")
print("Par segment :")
print(yearly_segments.round(2))

**Lecture :** la cible (`effective_growth`) oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.
L'écart entre contractuel et effectif suit la hausse du rabais : nul jusqu'en 2023, puis 1,2 point en 2024 et 3,0 points en 2025.
Les renouvellements à The Met n'existent qu'à partir de 2024 : pour les années antérieures, il n'y a pas d'historique ontarien.

## 2. Relations avec les données externes

Avant de les utiliser dans un modèle, on vérifie si les variables externes sont **réellement liées** à nos composantes.
Avec 5 à 7 années, ces relations sont **exploratoires** : elles orientent le modèle, elles ne prouvent rien.

### 2a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse observée - taux du TAL) serait stable d'une année à l'autre.

In [ ]:
renewal_vs_tal = pd.DataFrame({
    "renewal_QC_contract_growth": yearly_segments[("contract_growth", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
}).dropna()
renewal_vs_tal["premium_over_tal"] = renewal_vs_tal["renewal_QC_contract_growth"] - renewal_vs_tal["tal_rate"]
print(renewal_vs_tal.round(2))
print(f"\nCorrélation hausse des renouvellements QC / taux TAL : {renewal_vs_tal.iloc[:, :2].corr().iloc[0, 1]:.2f}")

**Lecture :** la relation est faible et la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas,
Équinoxe augmente bien au-dessus), puis proche de 0 depuis 2023 (+0,3, -1,1, +0,5). Depuis que le TAL est élevé, les
renouvellements québécois suivent à peu près le taux du TAL; avant, ils l'ignoraient. Le taux du TAL n'est qu'une
recommandation : il ne s'impose que si le locataire conteste. On ne l'impose donc pas comme ancrage :
le taux réglementaire entre dans le modèle comme **une variable parmi d'autres**, et le modèle apprend lui-même son poids.

### 2b. Hausse du rabais et inoccupation du marché (SCHL)

C'est l'hypothèse « vacance → concessions » testée sans succès avec les données internes (section 2g du notebook final),
reprise ici avec le taux d'inoccupation **du marché de Montréal** publié par la SCHL (enquête d'octobre).

In [ ]:
montreal = cmhc[cmhc["market"] == MONTREAL_MARKET].set_index("year")
discount_vs_vacancy = pd.DataFrame({
    "discount_change": yearly_target["discount_change"],
    "vacancy_rate": montreal["vacancy_rate"],
    "vacancy_change": montreal["vacancy_rate"].diff(),
}).dropna()
print(discount_vs_vacancy.round(2))
print(f"\nCorrélation hausse du rabais / variation de l'inoccupation (même année) : "
      f"{discount_vs_vacancy[['discount_change', 'vacancy_change']].corr().iloc[0, 1]:.2f}  (n = {len(discount_vs_vacancy)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(discount_vs_vacancy["vacancy_change"], discount_vs_vacancy["discount_change"])
for year, row in discount_vs_vacancy.iterrows():
    ax.annotate(str(year), (row["vacancy_change"], row["discount_change"]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("variation du taux d'inoccupation, Montréal (points, SCHL)")
ax.set_ylabel("hausse médiane du rabais (points)")
ax.set_title("Quand l'inoccupation du marché monte, les rabais augmentent")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** l'inoccupation à Montréal baisse jusqu'en 2023 (1,5 %, marché très serré), puis remonte (2,1 % en 2024,
2,9 % en 2025). Les rabais d'Équinoxe suivent : stables quand le marché se resserre, en forte hausse quand il se détend
(corrélation d'environ 0,8). C'est cohérent avec l'hypothèse de départ : **les concessions répondent au marché**.

**Prudence :** 4 points seulement, et la relation est **contemporaine** - l'inoccupation d'octobre de l'année *t* n'est
connue qu'à la fin de *t*. Elle ne peut donc pas servir directement à prévoir *t*; Dans le modèle, on teste l'inoccupation
d'octobre de l'année **précédente**, la seule connue au moment de prévoir (section 5).

## 3. Le modèle : gradient boosting au niveau des paires (scikit-learn)

**Pourquoi ce modèle :** notre définition est la médiane des hausses **de chaque bail** de l'année. Le modèle fait exactement
cela : il apprend la hausse d'une paire à partir de ce qu'on sait **avant** le nouveau bail, prédit chaque bail de l'année,
et prend la médiane. Il apprend sur environ 3 000 paires, et non sur 7 années.

**Variables (toutes connues avant le début du nouveau bail) :**

| Variable | Pourquoi |
|---|---|
| immeuble (indicatrices), chambres, superficie, étage | le produit loué |
| `gap_years` | durée écoulée depuis le bail précédent |
| `prev_discount_pct` | rabais du bail précédent : un rabais élevé peut reculer au bail suivant, et inversement |
| `regulatory_rate` | taux du TAL (Québec) ou ligne directrice (Ontario) de l'année, publiés avant l'année |
| `is_renewal` | renouvellement ou relocation |

**Le type de bail n'est pas connu d'avance** (le locataire restera-t-il?). Chaque bail est donc prédit deux fois, comme
renouvellement et comme relocation, pondéré par la part historique de renouvellements de son immeuble. La médiane se calcule
sur cet ensemble pondéré.

**Choix du modèle :** `HistGradientBoostingRegressor` avec une perte en erreur absolue (il vise la médiane, comme notre
définition), des arbres peu profonds et des feuilles d'au moins 40 paires pour limiter le surapprentissage.
L'inoccupation SCHL est testée en section 5, mais pas retenue (voir pourquoi en 5).

In [ ]:
BUILDINGS = sorted(leases["sBuilding"].unique())
SCHL_VACANCY = cmhc.set_index(["market", "year"])["vacancy_rate"]


def regulatory_rate_for(df):
    """Taux réglementaire de l'année du nouveau bail : TAL au Québec, ligne directrice en Ontario."""
    years = df["lease_start_year"]
    tal = years.map(regulatory_rates["tal_unheated_pct"])
    ontario = years.map(regulatory_rates["ontario_guideline_pct"])
    return np.where(df["sState"] == ONTARIO, ontario, tal)


def previous_october_vacancy(df):
    """Taux d'inoccupation SCHL d'octobre de l'année précédente, pour le marché de l'immeuble."""
    markets = np.where(df["sState"] == ONTARIO, OTTAWA_MARKET, MONTREAL_MARKET)
    return [SCHL_VACANCY.get((market, year - 1), np.nan) for market, year in zip(markets, df["lease_start_year"])]


def feature_matrix(df, use_prev_discount=True, use_regulatory=True, use_vacancy=False):
    features = df[UNIT_FEATURES].astype(float).copy()
    if use_prev_discount:
        features["prev_discount_pct"] = df["prev_discount_pct"]
    if use_regulatory:
        features["regulatory_rate"] = regulatory_rate_for(df)
    if use_vacancy:
        features["previous_october_vacancy"] = previous_october_vacancy(df)
    for building in BUILDINGS:
        features[f"building_{building}"] = (df["sBuilding"] == building).astype(float)
    return features


def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values, cumulative_weights = values[order], np.cumsum(weights[order])
    return sorted_values[np.searchsorted(cumulative_weights, cumulative_weights[-1] / 2)]


def predict_both_lease_types(model, population, training_pairs, feature_options):
    """Prédit chaque bail comme renouvellement et comme relocation, pondéré par la part de renouvellements de l'immeuble."""
    renewal_share = (population["sBuilding"].map(training_pairs.groupby("sBuilding")["is_renewal"].mean())
                     .fillna(training_pairs["is_renewal"].mean()).to_numpy())
    scenarios = []
    for status, weight in [(RENEWAL, renewal_share), (TURNOVER, 1 - renewal_share)]:
        scenario = population.assign(is_renewal=status)
        scenario["predicted_growth"] = model.predict(feature_matrix(scenario, **feature_options))
        scenario["weight"] = weight
        scenarios.append(scenario)
    return pd.concat(scenarios)


def fit_pair_model(training_pairs, feature_options):
    model = HistGradientBoostingRegressor(**GBM_PARAMS)
    model.fit(feature_matrix(training_pairs, **feature_options), training_pairs["effective_rent_growth_pct"])
    return model

### Population de baux à prévoir

- **Pour une année de backtest** : les baux qui ont réellement débuté cette année-là. On n'utilise que les informations de leur
  **bail précédent** (rabais, durée écoulée) et du logement, connues avant le début du nouveau bail.
- **Pour 2026** : le dernier bail connu de chaque logement se termine à une date connue (`lease_end`). Le prochain bail débute
  le lendemain. On retient les logements dont ce prochain bail débute en 2026, avec les mêmes filtres d'écart que pour l'historique.

In [ ]:
last_known_leases = leases.sort_values("lease_start").groupby(UNIT_KEY).tail(1).copy()
next_lease_start = last_known_leases["lease_end"] + pd.Timedelta(days=1)
population_2026 = last_known_leases.assign(
    lease_start_year=next_lease_start.dt.year,
    gap_years=(next_lease_start - last_known_leases["lease_start"]).dt.days / DAYS_PER_YEAR,
    prev_discount_pct=(1 - last_known_leases["effective_rent"] / last_known_leases["contract_rent"]) * PCT)
population_2026 = population_2026[
    (population_2026["lease_start_year"] == FORECAST_YEAR)
    & population_2026["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")]

print(f"{len(population_2026):,} logements dont le prochain bail débute en {FORECAST_YEAR}")
print(population_2026["sBuilding"].value_counts().to_string())
print(f"\nRabais moyen du bail précédent : population 2026 = {population_2026['prev_discount_pct'].mean():.1f} %, "
      f"baux de {FORECAST_YEAR - 1} = {pairs.loc[pairs['lease_start_year'] == FORECAST_YEAR - 1, 'prev_discount_pct'].mean():.1f} %")


def population_for(year):
    return population_2026 if year == FORECAST_YEAR else pairs[pairs["lease_start_year"] == year]


RETAINED_FEATURES = {"use_prev_discount": True, "use_regulatory": True, "use_vacancy": False}


def forecast_year(year, feature_options=RETAINED_FEATURES):
    """Entraîne sur les paires antérieures à `year`, prédit chaque bail de `year`, renvoie la médiane pondérée (%)."""
    training_pairs = pairs[pairs["lease_start_year"] < year]
    model = fit_pair_model(training_pairs, feature_options)
    predictions = predict_both_lease_types(model, population_for(year), training_pairs, feature_options)
    return weighted_median(predictions["predicted_growth"].to_numpy(), predictions["weight"].to_numpy())

## 4. Backtest

Règle : pour prévoir l'année *t*, le modèle n'utilise que ce qui était **connu au début de *t*** :

- il est entraîné uniquement sur les paires dont le nouveau bail a débuté **avant** *t*;
- pour les baux de *t*, il ne voit que le logement et le **bail précédent** (rabais, durée écoulée), pas le nouveau loyer;
- le type de bail (renouvellement ou relocation) n'est pas supposé connu : les deux cas sont pondérés (section 3);
- les taux réglementaires de *t* sont connus (publiés avant le début de l'année); la SCHL ne l'est que jusqu'à octobre *t* - 1.

Erreur = prévision - réel, en points de pourcentage. On la résume par l'**erreur absolue moyenne** (MAE) sur 2023-2025.

In [ ]:
def backtest(target_year, feature_options=RETAINED_FEATURES):
    """Prévoit target_year avec l'information disponible à la fin de target_year - 1, puis compare au réel."""
    forecast = forecast_year(target_year, feature_options)
    actual = yearly_target.loc[target_year, "effective_growth"]
    return {"forecast": forecast, "actual": actual, "error": forecast - actual}


backtest_results = pd.DataFrame({year: backtest(year) for year in BACKTEST_YEARS}).T
baseline_mae = backtest_results["error"].abs().mean()
print(backtest_results.round(2))
print(f"\nErreur absolue moyenne (MAE) du modèle de référence : {baseline_mae:.2f} point")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly_target.index, yearly_target["effective_growth"], "k-o", lw=2.5, label="réel (cible)")
ax.plot(backtest_results.index, backtest_results["forecast"], "--s", color="tab:blue", lw=2, label="prévision du modèle")
ax.axvspan(min(BACKTEST_YEARS) - 0.5, max(BACKTEST_YEARS) + 0.5, color="grey", alpha=0.1, label="période de backtest")
ax.set_xlabel("année")
ax.set_ylabel("croissance du loyer effectif, même unité (%)")
ax.set_title("Backtest : prévisions faites avec l'information disponible l'année précédente")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** le modèle se trompe en moyenne d'environ 1,2 point. Il suit bien 2024 et 2025, mais **rate 2023** :
il prévoit environ 4,5 % pour 2,0 % réel. Le ralentissement de 2023 arrive après deux années fortes (3,5 % et 4,6 %) et
rien dans l'historique ne l'annonçait. Avec 3 années de backtest seulement, l'erreur moyenne est indicative.

Pour comparaison, deux règles simples testées dans une version précédente faisaient moins bien : reprendre la hausse de l'année
précédente (erreur moyenne de 1,54 point) ou la moyenne des 3 dernières années (1,36 point).

## 5. Quelles variables comptent?

On refait le backtest et la prévision 2026 en retirant ou en ajoutant **un groupe de variables à la fois**.
Ça montre ce que chaque groupe apporte, et ce qui pousse la prévision 2026.

In [ ]:
FEATURE_VARIANTS = {
    "retenu (logement + rabais précédent + taux réglementaire)": RETAINED_FEATURES,
    "+ inoccupation SCHL": {"use_prev_discount": True, "use_regulatory": True, "use_vacancy": True},
    "sans rabais précédent": {"use_prev_discount": False, "use_regulatory": True, "use_vacancy": False},
    "sans taux réglementaire": {"use_prev_discount": True, "use_regulatory": False, "use_vacancy": False},
    "logement seulement": {"use_prev_discount": False, "use_regulatory": False, "use_vacancy": False},
}

variant_rows = {}
for variant_name, options in FEATURE_VARIANTS.items():
    row = {year: backtest(year, options)["forecast"] for year in BACKTEST_YEARS}
    row["MAE"] = np.mean([abs(row[year] - yearly_target.loc[year, "effective_growth"]) for year in BACKTEST_YEARS])
    row[FORECAST_YEAR] = forecast_year(FORECAST_YEAR, options)
    variant_rows[variant_name] = row
pd.DataFrame(variant_rows).T.round(2)

**Lecture :**

- **Le rabais du bail précédent est la variable la plus utile au backtest** : sans elle, l'erreur passe de 1,18 à 1,45 point.
  Le modèle a appris qu'après un rabais élevé, le loyer effectif du bail suivant monte davantage (le rabais tend à reculer),
  et qu'après un bail sans rabais, il monte peu (un rabais apparaît).
- **Mais c'est aussi elle qui pousse la prévision 2026 vers le haut.** Les logements qui renouvellent en 2026 partent d'un rabais
  record (9,0 % en moyenne, contre 5,8 % pour les baux de 2025) : le modèle **extrapole** au-delà de ce qu'il a vu et suppose
  implicitement que les rabais vont reculer. Sans cette variable, il prévoit 2,3 % au lieu de 3,8 %.
- **L'inoccupation SCHL n'améliore presque pas le backtest** (1,13 contre 1,18 point, seule 2025 change) mais fait bondir
  la prévision 2026 à 5,1 % : l'inoccupation d'octobre 2025 (2,9 % à Montréal) dépasse tout ce que le modèle a vu, et il
  n'a que 4 années pour en apprendre l'effet. Gain négligeable, extrapolation forte : on ne la retient pas.
- **Le taux réglementaire aide un peu** (1,18 contre 1,29 sans lui).

**En clair :** la prévision 2026 (3,8 %) repose sur l'hypothèse que les rabais records de 2025 vont partiellement
reculer. C'est plausible (le rabais d'un bail a tendance à revenir vers la moyenne), mais ce n'est pas garanti :
c'est la principale incertitude à traiter dans les prochaines versions.

## 6. Prévision 2026

On applique le modèle avec tout l'historique jusqu'à 2025 et les taux réglementaires 2026 (TAL : 3,1 %; Ontario : 2,1 %).
La prévision par immeuble et par nombre de chambres répond au bonus des consignes. Le modèle final est entraîné sur toutes les paires jusqu'en 2025, puis sauvegardé
(livrable « modèle entraîné »). Les effectifs indiquent combien de baux de 2026 entrent dans chaque médiane.

In [ ]:
def estimate_2026(feature_options=RETAINED_FEATURES):
    """Hausse 2026 du loyer effectif à unité constante (%), selon notre définition."""
    return forecast_year(FORECAST_YEAR, feature_options)


print(f"Hausse 2026 prévue (loyer effectif, même unité, médiane) : {estimate_2026():.2f} %\n")

final_training_pairs = pairs[pairs["lease_start_year"] < FORECAST_YEAR]
final_model_d = fit_pair_model(final_training_pairs, RETAINED_FEATURES)
predictions_2026 = predict_both_lease_types(final_model_d, population_2026, final_training_pairs, RETAINED_FEATURES)


def weighted_median_by(df, group_column):
    rows = {}
    for group_value, group in df.groupby(group_column):
        rows[group_value] = {
            "baux_2026": len(group) // 2,   # chaque bail apparaît deux fois (renouvellement et relocation)
            "hausse_effective_prevue_pct": weighted_median(group["predicted_growth"].to_numpy(),
                                                           group["weight"].to_numpy()),
        }
    return pd.DataFrame(rows).T


print("Par immeuble :")
print(weighted_median_by(predictions_2026, "sBuilding").round(2), end="\n\n")
print("Par nombre de chambres :")
print(weighted_median_by(predictions_2026, "sBeds").round(2))

MODEL_D_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model_d, "features": RETAINED_FEATURES, "buildings": BUILDINGS}, MODEL_D_PATH)
print(f"\nModèle sauvegardé : {MODEL_D_PATH}")

## 7. Bilan du modèle de référence et pistes d'amélioration

**Résultat de référence :** hausse 2026 prévue de **3,8 %** (loyer effectif, même unité, médiane des baux 2026),
avec une erreur moyenne de **1,18 point** au backtest 2023-2025. Par immeuble : 3,4 % à The Met, 3,6 à 3,9 % ailleurs.

**Forces :**
- il suit littéralement notre définition (médiane des hausses de chaque bail);
- il tient compte de la composition réelle des baux de 2026 (937 logements connus d'avance);
- il donne la prévision par immeuble et par nombre de chambres;
- chaque variable est connue avant le début du bail : pas d'information future dans le backtest.

**Faiblesses à corriger :**
1. **2023 est mal prévu** (4,5 % pour 2,0 %) : le modèle ne voit pas venir un ralentissement général.
2. **La prévision 2026 dépend d'une hypothèse cachée sur les rabais** : partis d'un niveau record (9 %), ils reculeraient en
   partie. Sans la variable « rabais précédent », la prévision tombe à 2,3 %.
3. **Pas d'intervalle d'incertitude** : un seul chiffre.
4. **Peu d'années pour apprendre les effets d'année** (taux réglementaire, inoccupation).

**Pistes d'amélioration à tester, une à la fois, sur le même backtest :**
- borner ou transformer le rabais précédent pour limiter l'extrapolation hors de la plage d'entraînement;
- ajouter des variables de position du loyer (écart au loyer affiché, loyer au pi² par rapport à l'immeuble);
- donner plus de poids aux années récentes à l'entraînement;
- ajuster les hyperparamètres (profondeur, taille des feuilles) par validation temporelle;
- construire un intervalle de prévision à partir des erreurs du backtest ou par quantiles (`loss="quantile"`);
- confirmer l'exemption de The Met (première occupation après le 15 novembre 2018).